# Day 5 Addendum — Test Set Evaluation

Validation results alone can be misleading -- this addendum evaluates the trained iccad1 and iccad3 checkpoints against their actual, untouched test sets (never seen during training or validation), including a leakage-aware "clean test" subset per Day 1's base-pattern overlap finding.

## Method

For each benchmark: load the best checkpoint, build `test_df` from the `test/` folder, and build `clean_test_df` by removing any test file whose augmented base pattern also appears among the benchmark's *training* files (`fold != 0`). Both are evaluated at the default 0.5 threshold using the same precision/recall/mean-prediction-by-label diagnostics used during validation.

## iccad1 test results

1065 of 4905 test files (21.7%) removed for the clean subset -- consistent with, though smaller than, Day 1's original full-training-set overlap estimate (that check used the full training set; here only the `fold != 0` portion was checked, since that's what the model actually trained on).

| | HS mean | NHS mean | precision | recall |
|---|---|---|---|---|
| Full test (n=226 HS / 4679 NHS) | 0.2480 | 0.2466 | 0.000 | 0.000 |
| Clean test (n=147 HS / 3693 NHS) | 0.2475 | 0.2462 | 0.000 | 0.000 |

**The validation-time collapse reproduces identically on test.** HS and NHS prediction means differ by ~0.001, even smaller than validation's already-tiny 0.006 gap -- every single hotspot in both test sets falls below the 0.5 threshold. This rules out "validation set was just small/unlucky" as an explanation; the model genuinely failed to learn any transferable signal from iccad1's ~350 training images.

Notably, full-test and clean-test results are nearly identical here -- unlike iccad3 below. This makes sense: leakage only inflates results for a model that's exploiting pattern-level memorization to get correct answers. This model never learned anything correct enough to exploit in the first place, so removing the "cheatable" overlapping patterns changes almost nothing.

## iccad3 test results

| | HS mean | NHS mean | precision | recall |
|---|---|---|---|---|
| Validation (n=182 HS / 929 NHS) | 0.9632 | 0.0082 | 0.994 | 0.967 |
| Full test, threshold=0.5 (n=1808 HS / 46333 NHS) | 0.9657 | 0.2685 | 0.128 | 0.970 |
| Clean test, threshold=0.5 (n=1081 HS / 37748 NHS) | 0.9613 | 0.2072 | 0.125 | 0.965 |
| Full test, threshold=0.7 (val-selected) | -- | -- | 0.132 | 0.960 |

**A real, diagnosed distribution-shift problem, not overfitting.** Precision collapses from 0.994 (validation) to ~0.13 (test) while recall barely moves (0.967 -> 0.970). Ruled out as simple overfitting, since an overfit model would also perform poorly on validation -- it doesn't; validation genuinely shows strong, well-separated (not memorized) predictions.

**Root cause identified: train/test hotspot-rate mismatch.** iccad3's training data is 16.37% HS; its test data is only 3.75% HS (Day 2 EDA numbers). The model's implicit calibration reflects the training-time base rate; on test, that same behavior applied to a much larger pool of true negatives produces a much larger absolute count of false positives (NHS mean prediction jumps from 0.008 on val to 0.27 on test -- roughly 30x higher).

**Threshold tuning cannot fix this.** A threshold sweep on validation alone (methodologically correct -- never tuned against test directly) showed precision already near-maximal (0.994) and largely threshold-insensitive from 0.5-0.9. Applying the same val-selected threshold (0.7) to test moved precision only from 0.128 to 0.132 -- confirming the problem is a genuine shift in the underlying probability distribution, not a miscalibrated cutoff that a different threshold could resolve.

**Open question for Day 6-7:** does focal loss, by construction, produce a model whose calibration is less tied to the exact training-time base rate -- and does that make it more robust to this kind of train/test distribution shift, or is this an orthogonal problem focal loss won't touch? Worth testing explicitly rather than assuming either way.

## Housekeeping

Model checkpoints (`.keras` files) were removed from git tracking and added to `.gitignore` -- they remain available on Google Drive (`/content/drive/MyDrive/iccad_checkpoints/`), but committing large binary artifacts to the repo was bloating its size without benefit, since they're build outputs rather than source code.

**Next:** train iccad2, iccad4, and iccad5 using the same `run_baseline()` function, then run this same test-evaluation pattern (full + clean, with a val-selected threshold check) against all three before moving to Day 6-7's focal loss implementation.